In [1]:
!git clone https://github.com/dinhngoctoan/DATN_DTA

Cloning into 'DATN_DTA'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 103 (delta 51), reused 56 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 5.00 MiB | 10.32 MiB/s, done.
Resolving deltas: 100% (51/51), done.


In [2]:
%cd DATN_DTA


/kaggle/working/DATN_DTA


In [3]:
%cd data


/kaggle/working/DATN_DTA/data


In [4]:
%mkdir processed


In [5]:
!cp -r /kaggle/input/ecfp-davis/data/processed/* /kaggle/working/DATN_DTA/data/processed


In [6]:
%cd /kaggle/working/DATN_DTA


/kaggle/working/DATN_DTA


In [7]:
!pip install rdkit-pypi
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install torch-geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.4/29.4 MB 50.7 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 30.6 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 24.2 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 22.7 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.8/891.8 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:00


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU
from torch_geometric.nn import GATConv, GCNConv, GINConv
from torch_geometric.nn import global_add_pool
from torch_geometric.nn import global_mean_pool as gap
from torch_geometric.nn import global_max_pool as gmp


class MoEFFN(nn.Module):
    """
    Mixture-of-Experts feed-forward block (Top-K gating + load-balancing loss),
    replaces Linear -> ReLU -> Dropout -> Linear -> ReLU -> Dropout.
    """
    def __init__(self, in_dim, hidden_dim, out_dim, num_experts=4, top_k=2, dropout=0.2):
        super(MoEFFN, self).__init__()
        assert 1 <= top_k <= num_experts, "top_k must be in [1, num_experts]"
        self.num_experts = num_experts
        self.top_k = top_k
        self.out_dim = out_dim

        self.gate = nn.Linear(in_dim, num_experts)

        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
            )
            for _ in range(num_experts)
        ])

    def forward(self, x):
        """
        x: [batch_size, in_dim]
        return:
            out: [batch_size, out_dim]
            aux_loss: scalar tensor (load-balancing loss)
        """
        batch_size = x.size(0)
        gate_logits = self.gate(x)                      # [B, num_experts]
        gate_probs = F.softmax(gate_logits, dim=-1)      # [B, num_experts]

        topk_val, topk_idx = gate_probs.topk(self.top_k, dim=-1)   # [B, top_k]
        topk_weight = topk_val / (topk_val.sum(dim=-1, keepdim=True) + 1e-9)

        out = torch.zeros(batch_size, self.out_dim, device=x.device, dtype=x.dtype)

        for k in range(self.top_k):
            expert_idx_k = topk_idx[:, k]
            weight_k = topk_weight[:, k]
            for e in range(self.num_experts):
                mask = (expert_idx_k == e)
                if mask.any():
                    out[mask] += weight_k[mask].unsqueeze(-1) * self.experts[e](x[mask])

        density = gate_probs.mean(dim=0)
        importance = gate_probs.sum(dim=0)
        aux_loss = self.num_experts * (density * F.softmax(importance, dim=-1)).sum()

        return out, aux_loss


# GAT model
class GATnGCN_Model(torch.nn.Module):
    def __init__(self, num_features_xd=78, n_output=1, num_features_xt=41, num_features_xt_seq=25,
                 output_dim=128, dropout=0.2, embed_dim=128, n_filters=32,
                 moe_num_experts=4, moe_top_k=2,
                 moe_drug_num_experts=3, moe_drug_top_k=2):
        super(GATnGCN_Model, self).__init__()
        dim = 32
        # drug
        self.conv1 = GATConv(num_features_xd, num_features_xd, heads=10)
        self.conv2 = GCNConv(num_features_xd * 10, num_features_xd * 10)

        # ===== MoE thay cho fc_g1 -> relu -> dropout -> fc_g2 (drug graph readout) =====
        # Use fewer experts than the final fusion block because this block is closer to the input
        # and has less fused information, reducing the risk of overfitting.
        self.moe_drug = MoEFFN(in_dim=num_features_xd * 10 * 2, hidden_dim=1500, out_dim=output_dim,
                                num_experts=moe_drug_num_experts, top_k=moe_drug_top_k, dropout=dropout)

        # ecfp
        self.fc_ecfp = nn.Linear(2048, 128)
        self.bn_ecfp = nn.BatchNorm1d(128)
        # 3D protein
        self.conv_prot1 = GCNConv(num_features_xt, num_features_xt)
        self.conv_prot2 = GCNConv(num_features_xt, 64)
        self.lin1 = nn.Linear(64, 128)

        # 1D protein
        self.embedding_xt = nn.Embedding(num_features_xt_seq + 1, embed_dim)
        self.conv_xt_1 = nn.Conv1d(in_channels=1000, out_channels=n_filters, kernel_size=8)
        self.pool_xt_1 = nn.MaxPool1d(2, ceil_mode=True)
        self.conv_xt_2 = nn.Conv1d(in_channels=n_filters, out_channels=n_filters * 2, kernel_size=8)
        self.pool_xt_2 = nn.MaxPool1d(2, ceil_mode=True)
        self.conv_xt_3 = nn.Conv1d(in_channels=n_filters * 2, out_channels=n_filters * 4, kernel_size=8)
        self.pool_xt_3 = nn.MaxPool1d(2, ceil_mode=True)
        self.fc_xt = nn.Linear(10 * 128, 128)

        # cross-attention
        self.drug_protein_attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=dropout)
        self.protein_drug_attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, dropout=dropout)

        # concat
        self.fc_xd = nn.Linear(256, 128)
        self.fc_xp = nn.Linear(256, 128)

        # Replace fc1 -> relu -> dropout -> fc2 -> relu -> dropout with MoE for final fusion.
        # Block 1: 256 (xd 128 + xp 128) -> 1024
        self.moe1 = MoEFFN(in_dim=128 * 2, hidden_dim=1024, out_dim=1024,
                            num_experts=moe_num_experts, top_k=moe_top_k, dropout=dropout)
        # Block 2: 1024 -> 256
        self.moe2 = MoEFFN(in_dim=1024, hidden_dim=512, out_dim=256,
                            num_experts=moe_num_experts, top_k=moe_top_k, dropout=dropout)

        self.out = nn.Linear(256, n_output)

        # activation and regularization
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, data_drug, data_protein):
        # graph input feed-forward
        x_drug, edge_index_drug, batch_drug = data_drug.x, data_drug.edge_index, data_drug.batch

        x_drug = self.conv1(x_drug, edge_index_drug)
        x_drug = self.relu(x_drug)
        x_drug = self.conv2(x_drug, edge_index_drug)
        x_drug = self.relu(x_drug)
        # apply global max pooling (gmp) and global mean pooling (gap)
        x_drug = torch.cat([gmp(x_drug, batch_drug), gap(x_drug, batch_drug)], dim=1)

        # ===== MoE thay cho fc_g1 -> relu -> dropout -> fc_g2 =====
        # Each MoEFFN expert already includes activation and dropout,
        # so no additional relu/dropout is needed here.
        x_drug, aux_loss_drug = self.moe_drug(x_drug)

        # # ecfp
        # x_drug_ecfp = data_drug.ecfp
        # x_drug_ecfp = self.fc_ecfp(x_drug_ecfp)
        # x_drug_ecfp = self.bn_ecfp(x_drug_ecfp)
        # x_drug_ecfp = self.relu(x_drug_ecfp)
        # x_drug_ecfp = self.dropout(x_drug_ecfp)

        # protein input feed-forward:
        x_prots, edge_index_prots, prots_batch = data_protein.x, data_protein.edge_index, data_protein.batch
        # 3D process
        x_prots = F.dropout(x_prots, p=0.2, training=self.training)
        x_prots = self.relu(self.conv_prot1(x_prots, edge_index_prots))
        x_prots = F.dropout(x_prots, p=0.2, training=self.training)
        x_prots = self.relu(self.conv_prot2(x_prots, edge_index_prots))
        x_prots = gmp(x_prots, prots_batch)
        x_prots = self.lin1(x_prots)
        x_prots = self.relu(x_prots)

        # # 1D process
        # prot_seq = data_drug.protein_seq
        # embedded_x_seq = self.embedding_xt(prot_seq)
        # x_prots_seq = self.conv_xt_1(embedded_x_seq)
        # x_prots_seq = F.relu(x_prots_seq)
        # x_prots_seq = self.pool_xt_1(x_prots_seq)
        # x_prots_seq = self.conv_xt_2(x_prots_seq)
        # x_prots_seq = F.relu(x_prots_seq)
        # x_prots_seq = self.pool_xt_2(x_prots_seq)
        # x_prots_seq = self.conv_xt_3(x_prots_seq)
        # x_prots_seq = F.relu(x_prots_seq)
        # x_prots_seq = self.pool_xt_3(x_prots_seq)

        # # flatten
        # x_prots_seq = x_prots_seq.view(-1, 10 * 128)
        # x_prots_seq = self.fc_xt(x_prots_seq)

        # cross-attention
        x_drug = x_drug.unsqueeze(0)
        x_prots = x_prots.unsqueeze(0)

        # Drug to protein attention
        x_drug_attended, _ = self.drug_protein_attention(x_drug, x_prots, x_prots)
        x_drug_attended = x_drug_attended.squeeze(0)

        # Protein to drug attention
        x_prots_attended, _ = self.protein_drug_attention(x_prots, x_drug, x_drug)
        x_prots_attended = x_prots_attended.squeeze(0)

        # Concatenate four sources: drug graph + ECFP, protein 3D graph + protein sequence.
        # xd = torch.cat((x_drug_attended, x_drug_ecfp), 1)   # drug graph + ecfp
        # xp = torch.cat((x_prots_attended, x_prots_seq), 1)  # protein 3D + protein sequence
        # xd = self.fc_xd(xd)
        # xd = self.relu(xd)
        # xp = self.fc_xp(xp)
        # xp = self.relu(xp)

        xc = torch.cat((x_drug_attended, x_prots_attended), 1)  # [batch_size, 256]

        # MoE dense layers replace fc1/fc2 for final fusion.
        xc, aux_loss1 = self.moe1(xc)     # [batch_size, 1024]
        xc = self.relu(xc)
        xc = self.dropout(xc)

        xc, aux_loss2 = self.moe2(xc)     # [batch_size, 256]
        xc = self.relu(xc)
        xc = self.dropout(xc)

        out = self.out(xc)

        # Store the combined auxiliary loss from drug readout and final fusion as an attribute
        # instead of returning a tuple, preserving the existing interface: train/predict loops
        # call model(...) and consume its output directly.
        # Read model.aux_loss after calling model(...).
        self.aux_loss = aux_loss_drug + aux_loss1 + aux_loss2
        return out

/usr/local/lib/python3.11/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  import torch_geometric.typing
/usr/local/lib/python3.11/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /usr/local/lib/python3.11/dist-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  import torch_geometric.typing


In [9]:
import numpy as np
import pandas as pd
import sys, os
from random import shuffle
import torch
import torch.nn as nn
#from models.gat_gcn import GAT_GCNNet
#from models.gin_gcn import GIN_GCNNet
from utils import *
import copy
# training function at each epoch
def train(model,device, train_loader_drug,train_loader_protein, optimizer, epoch):
    print('Training on {} samples...'.format(len(train_loader_drug.dataset)))
    model.train()
    for batch_idx, (data_drug,data_protein) in enumerate(zip(train_loader_drug,train_loader_protein)):
        data_drug = data_drug.to(device)
        data_protein = data_protein.to(device)
        optimizer.zero_grad()
        output = model(data_drug,data_protein)
        
        pred_loss = loss_fn(output, data_drug.y.view(-1, 1).float().to(device))
        aux_loss = getattr(model, 'aux_loss', 0.0)
        loss = pred_loss + AUX_LOSS_WEIGHT * aux_loss
        
        loss.backward()
        optimizer.step()
        if batch_idx % LOG_INTERVAL == 0:
            print('Train epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(epoch,
                                                                           batch_idx * len(data_drug.x),
                                                                           len(train_loader_drug.dataset),
                                                                           100. * batch_idx / len(train_loader_drug),
                                                                           loss.item()))

def predicting(model, device, loader_drug,loader_protein):
    model.eval()
    total_preds = torch.Tensor()
    total_labels = torch.Tensor()
    print('Make prediction for {} samples...'.format(len(loader_drug.dataset)))
    with torch.no_grad():
        for data_drug,data_protein in zip(loader_drug,loader_protein):
            data_drug = data_drug.to(device)
            data_protein = data_protein.to(device)
            output = model(data_drug,data_protein)
            total_preds = torch.cat((total_preds, output.cpu()), 0)
            total_labels = torch.cat((total_labels, data_drug.y.view(-1, 1).cpu()), 0)
    return total_labels.numpy().flatten(),total_preds.numpy().flatten()


datasets = ['davis']#[['davis','kiba'][int(sys.argv[1])]]
modeling = GATnGCN_Model#[GIN_GCNNet, GAT_GCNNet][int(sys.argv[2])]
model_st = modeling.__name__

cuda_name = "cuda:0"
#if len(sys.argv)>3:
#    cuda_name = "cuda:" + str(int(sys.argv[3]))
print('cuda_name:', cuda_name)

TRAIN_BATCH_SIZE = 512
TEST_BATCH_SIZE = 512
LR = 0.0005
LOG_INTERVAL = 20
NUM_EPOCHS = 1000
AUX_LOSS_WEIGHT = 0 # Weight for the MoE load-balancing loss.

print('Learning rate: ', LR)
print('Epochs: ', NUM_EPOCHS)

# Main program: iterate over different datasets
for dataset in datasets:
    print('\nrunning on ', model_st + '_' + dataset )
    processed_data_file_train_drug = 'data/processed/' + dataset + '_train_drug.pt'
    processed_data_file_test_drug = 'data/processed/' + dataset + '_test_drug.pt'
    processed_data_file_train_protein = 'data/processed/' + dataset + '_train_protein.pt'
    processed_data_file_test_protein = 'data/processed/' + dataset + '_test_protein.pt'
    if ((not os.path.isfile(processed_data_file_train_drug)) or (not os.path.isfile(processed_data_file_test_drug)) or (not os.path.isfile(processed_data_file_train_protein)) or (not os.path.isfile(processed_data_file_test_protein))):
        print('please run create_data.py to prepare data in pytorch format!')
    else:
        train_data_drug = TestbedDataset(root='data', dataset=dataset+'_train_drug')
        test_data_drug = TestbedDataset(root='data', dataset=dataset+'_test_drug')
        train_data_protein = TestbedDataset(root='data', dataset=dataset+'_train_protein')
        test_data_protein = TestbedDataset(root='data', dataset=dataset+'_test_protein')

        # make data PyTorch mini-batch processing ready
        train_loader_drug = DataLoader(train_data_drug, batch_size=TRAIN_BATCH_SIZE, shuffle=False)
        test_loader_drug = DataLoader(test_data_drug, batch_size=TEST_BATCH_SIZE, shuffle=False)
        train_loader_protein = DataLoader(train_data_protein, batch_size=TRAIN_BATCH_SIZE, shuffle=False)
        test_loader_protein = DataLoader(test_data_protein, batch_size=TEST_BATCH_SIZE, shuffle=False)

        # training the model
        device = torch.device(cuda_name if torch.cuda.is_available() else "cpu")
        model = modeling().to(device)
        loss_fn = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        best_mse = 1000
        best_ci = 0
        best_epoch = -1
        model_file_name = 'model_' + model_st + '_' + dataset +  '.model'
        result_file_name = 'result_' + model_st + '_' + dataset +  '.csv'
        for epoch in range(NUM_EPOCHS):
            train(model, device, train_loader_drug,train_loader_protein, optimizer, epoch+1)
            G,P = predicting(model, device, test_loader_drug,test_loader_protein)
            ret = [rmse(G,P),mse(G,P),pearson(G,P),spearman(G,P),ci(G,P)]
            if ret[1]<best_mse:
                torch.save(model.state_dict(), model_file_name)
                with open(result_file_name,'w') as f:
                    f.write(','.join(map(str,ret)))
                best_epoch = epoch+1
                best_mse = ret[1]
                best_ci = ret[-1]
                best_model_state = copy.deepcopy(model.state_dict())  
                print('rmse improved at epoch ', best_epoch, '; best_mse,best_ci:', best_mse,best_ci,model_st,dataset)
            else:
                print(ret[1],ret[-1],'No improvement since epoch ', best_epoch, '; best_mse,best_ci:', best_mse,model_st,dataset)

            if (epoch + 1) in [500, 700, 1000] and best_model_state is not None:
                torch.save(best_model_state, f"{model_file_name}_best_until_epoch_{epoch+1}.pt")
                print(f"==> Saved best-so-far model at epoch {epoch+1} (best at epoch {best_epoch})")

cuda_name: cuda:0
Learning rate:  0.0005
Epochs:  1000

running on  GATnGCN_Model_davis
data/processed/davis_train_drug.pt exists
data/processed/davis_test_drug.pt exists
data/processed/davis_train_protein.pt exists
data/processed/davis_test_protein.pt exists


/tmp/ipykernel_19/1324035657.py:84: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader_drug = DataLoader(train_data_drug, batch_size=TRAIN_BATCH_SIZE, shuffle=False)
/tmp/ipykernel_19/1324035657.py:85: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader_drug = DataLoader(test_data_drug, batch_size=TEST_BATCH_SIZE, shuffle=False)
/tmp/ipykernel_19/1324035657.py:86: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader_protein = DataLoader(train_data_protein, batch_size=TRAIN_BATCH_SIZE, shuffle=False)
/tmp/ipykernel_19/1324035657.py:87: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader_protein = DataLoader(test_data_protein, batch_size=TEST_BATCH_SIZE, shuffle=False)


Training on 25046 samples...


/usr/local/lib/python3.11/dist-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Train epoch: 1 [0/25046 (0%)]	Loss: 29.052967
Train epoch: 1 [327100/25046 (41%)]	Loss: 1.754195
Train epoch: 1 [653720/25046 (82%)]	Loss: 1.143362
Make prediction for 5010 samples...
rmse improved at epoch  1 ; best_mse,best_ci: 1.7213979 0.5467717487873756 GATnGCN_Model davis
Training on 25046 samples...
Train epoch: 2 [0/25046 (0%)]	Loss: 0.912929
Train epoch: 2 [327100/25046 (41%)]	Loss: 0.893691
Train epoch: 2 [653720/25046 (82%)]	Loss: 0.886555
Make prediction for 5010 samples...
rmse improved at epoch  2 ; best_mse,best_ci: 1.3221998 0.5663193879162249 GATnGCN_Model davis
Training on 25046 samples...
Train epoch: 3 [0/25046 (0%)]	Loss: 0.830205
Train epoch: 3 [327100/25046 (41%)]	Loss: 0.915477
Train epoch: 3 [653720/25046 (82%)]	Loss: 0.848079
Make prediction for 5010 samples...
rmse improved at epoch  3 ; best_mse,best_ci: 1.2171701 0.5918258139719023 GATnGCN_Model davis
Training on 25046 samples...
Train epoch: 4 [0/25046 (0%)]	Loss: 0.747011
Train epoch: 4 [327100/25046 (41%